In [1]:
# Basic libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Load required datasets
claims = pd.read_csv(DATA_DIR / "claims.csv")
policies = pd.read_csv(DATA_DIR / "policies.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")

print("Claims shape:", claims.shape)

Claims shape: (75000, 14)


In [2]:
# Convert claim dates to datetime
claims["INCIDENT_DATE"] = pd.to_datetime(
    claims["INCIDENT_DATE"],
    errors="coerce"
)

claims["REPORTED_DATE"] = pd.to_datetime(
    claims["REPORTED_DATE"],
    errors="coerce"
)

In [3]:
# Binary fraud target
claims["TARGET_FRAUD"] = np.where(
    claims["FRAUD_FLAG"] == "No",
    0,
    1
)

# Check class distribution
print(claims["TARGET_FRAUD"].value_counts())
print(claims["TARGET_FRAUD"].value_counts(normalize=True) * 100)

TARGET_FRAUD
0    72841
1     2159
Name: count, dtype: int64
TARGET_FRAUD
0    97.121333
1     2.878667
Name: proportion, dtype: float64


In [5]:
# Number of days between incident and claim reporting
claims["REPORTING_DELAY_DAYS"] = (
    claims["REPORTED_DATE"] -
    claims["INCIDENT_DATE"]
).dt.days

# Extract month of incident
claims["INCIDENT_MONTH"] = claims["INCIDENT_DATE"].dt.month

In [6]:
# Policy features available around claim time
policy_features = policies[
    [
        "POLICY_ID",
        "POLICY_TYPE",
        "SUM_INSURED",
        "ANNUAL_PREMIUM",
        "PAYMENT_MODE",
        "RISK_SCORE",
        "RISK_BAND"
    ]
].copy()

# Add policy information to each claim
fraud_df = claims.merge(
    policy_features,
    on="POLICY_ID",
    how="left",
    validate="many_to_one"
)

In [7]:
# Customer-level features
customer_features = customers[
    [
        "CUSTOMER_ID",
        "AGE",
        "GENDER",
        "MARITAL_STATUS",
        "OCCUPATION",
        "ANNUAL_INCOME",
        "STATE",
        "CREDIT_SCORE",
        "CUSTOMER_RISK_SEGMENT"
    ]
].copy()

# Add customer information
fraud_df = fraud_df.merge(
    customer_features,
    on="CUSTOMER_ID",
    how="left",
    validate="many_to_one"
)

In [8]:
# Numerical features available before final fraud outcome
numeric_features = [
    "CLAIM_AMOUNT",
    "REPORTING_DELAY_DAYS",
    "INCIDENT_MONTH",
    "AGE",
    "ANNUAL_INCOME",
    "CREDIT_SCORE",
    "SUM_INSURED",
    "ANNUAL_PREMIUM",
    "RISK_SCORE"
]

# Categorical features
categorical_features = [
    "CLAIM_TYPE",
    "SOURCE",
    "CLAIM_SEVERITY",
    "GENDER",
    "MARITAL_STATUS",
    "OCCUPATION",
    "STATE",
    "CUSTOMER_RISK_SEGMENT",
    "POLICY_TYPE",
    "PAYMENT_MODE",
    "RISK_BAND"
]

all_features = numeric_features + categorical_features

In [9]:
# Ensure all selected features exist
missing_features = [
    col for col in all_features
    if col not in fraud_df.columns
]

print("Missing features:", missing_features)

Missing features: []


In [10]:
# Sort claims by reported date
fraud_df["REPORTED_DATE"] = pd.to_datetime(
    fraud_df["REPORTED_DATE"],
    errors="coerce"
)

fraud_df = fraud_df.sort_values(
    "REPORTED_DATE"
).reset_index(drop=True)

In [12]:
# X = input features
X = fraud_df[all_features].copy()

# y = target
y = fraud_df["TARGET_FRAUD"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (75000, 20)
y shape: (75000,)


In [13]:
# First 80% = training data
# Latest 20% = testing data
split_index = int(len(fraud_df) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (60000, 20)
Test : (15000, 20)


In [14]:
# Check class distribution
print("Training target distribution:")
print(y_train.value_counts(normalize=True) * 100)

Training target distribution:
TARGET_FRAUD
0    97.085
1     2.915
Name: proportion, dtype: float64


In [15]:
# Import preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [16]:
# Numeric preprocessing:
# missing values -> median
# numeric values -> standard scaling
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [17]:
# Categorical preprocessing:
# missing values -> most frequent category
# categories -> one-hot encoded columns
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [18]:
# Combine numeric and categorical preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [19]:
# Import Logistic Regression
from sklearn.linear_model import LogisticRegression

# Build preprocessing + Logistic Regression pipeline
fraud_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [21]:
# Train fraud model
fraud_lr.fit(
    X_train,
    y_train
)

print("Fraud Logistic Regression trained successfully")

Fraud Logistic Regression trained successfully


In [22]:
# Predict fraud class
lr_pred = fraud_lr.predict(X_test)

# Predict fraud probability
lr_prob = fraud_lr.predict_proba(X_test)[:, 1]

In [24]:
# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Calculate metrics
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_prob)
lr_pr_auc = average_precision_score(y_test, lr_prob)

print("Accuracy :", round(lr_accuracy, 4))
print("Precision:", round(lr_precision, 4))
print("Recall   :", round(lr_recall, 4))
print("F1 Score :", round(lr_f1, 4))
print("ROC-AUC  :", round(lr_auc, 4))
print("PR-AUC   :", round(lr_pr_auc, 4))

Accuracy : 0.6203
Precision: 0.033
Recall   : 0.4561
F1 Score : 0.0616
ROC-AUC  : 0.5616
PR-AUC   : 0.0415


In [25]:
# Class-wise fraud performance
print(
    classification_report(
        y_test,
        lr_pred,
        target_names=["Normal", "Fraud"]
    )
)

              precision    recall  f1-score   support

      Normal       0.98      0.62      0.76     14590
       Fraud       0.03      0.46      0.06       410

    accuracy                           0.62     15000
   macro avg       0.50      0.54      0.41     15000
weighted avg       0.95      0.62      0.74     15000



In [26]:
# Import Random Forest classifier
from sklearn.ensemble import RandomForestClassifier

# Build preprocessing + Random Forest pipeline
fraud_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("model", RandomForestClassifier(
            n_estimators=300,        # Number of trees
            max_depth=12,            # Maximum depth of each tree
            min_samples_split=10,    # Minimum rows required to split
            min_samples_leaf=5,      # Minimum rows in a leaf
            class_weight="balanced", # Handle fraud class imbalance
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [27]:
# Import Random Forest classifier
from sklearn.ensemble import RandomForestClassifier

# Build preprocessing + Random Forest pipeline
fraud_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("model", RandomForestClassifier(
            n_estimators=300,        # Number of trees
            max_depth=12,            # Maximum depth of each tree
            min_samples_split=10,    # Minimum rows required to split
            min_samples_leaf=5,      # Minimum rows in a leaf
            class_weight="balanced", # Handle fraud class imbalance
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [28]:
# Train Random Forest fraud model
fraud_rf.fit(
    X_train,
    y_train
)

print("Fraud Random Forest trained successfully")

Fraud Random Forest trained successfully


In [29]:
# Predict final fraud class
rf_pred = fraud_rf.predict(X_test)

# Predict probability of fraud
rf_prob = fraud_rf.predict_proba(X_test)[:, 1]

In [30]:
# Calculate Random Forest metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)
rf_pr_auc = average_precision_score(y_test, rf_prob)

print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("ROC-AUC  :", round(rf_auc, 4))
print("PR-AUC   :", round(rf_pr_auc, 4))

Accuracy : 0.9255
Precision: 0.0438
Recall   : 0.0829
F1 Score : 0.0573
ROC-AUC  : 0.5654
PR-AUC   : 0.036


In [31]:
# Class-wise fraud performance
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=["Normal", "Fraud"]
    )
)

              precision    recall  f1-score   support

      Normal       0.97      0.95      0.96     14590
       Fraud       0.04      0.08      0.06       410

    accuracy                           0.93     15000
   macro avg       0.51      0.52      0.51     15000
weighted avg       0.95      0.93      0.94     15000



In [33]:
# Compare Logistic Regression and Random Forest
fraud_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy": [
        lr_accuracy,
        rf_accuracy
    ],

    "Precision": [
        lr_precision,
        rf_precision
    ],

    "Recall": [
        lr_recall,
        rf_recall
    ],

    "F1 Score": [
        lr_f1,
        rf_f1
    ],

    "ROC-AUC": [
        lr_auc,
        rf_auc
    ],

    "PR-AUC": [
        lr_pr_auc,
        rf_pr_auc
    ]
})

fraud_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,Logistic Regression,0.620267,0.033039,0.456098,0.061614,0.561648,0.041498
1,Random Forest,0.925467,0.043814,0.082927,0.057336,0.565399,0.036013


In [41]:
# Import XGBoost classifier
from xgboost import XGBClassifier

# Create XGBoost pipeline
fraud_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("model", XGBClassifier(
            n_estimators=300,      # Number of boosting trees
            max_depth=5,           # Maximum depth of each tree
            learning_rate=0.05,    # Learning speed
            subsample=0.8,         # Use 80% rows per tree
            colsample_bytree=0.8,  # Use 80% features per tree
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [49]:
# Create balanced sample weights
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

In [50]:
# Train fraud XGBoost model
fraud_xgb.fit(
    X_train,
    y_train,
    model__sample_weight=sample_weights
)

print("Fraud XGBoost trained successfully")

Fraud XGBoost trained successfully


In [51]:
# Predict fraud class
xgb_pred = fraud_xgb.predict(X_test)

# Predict fraud probability
xgb_prob = fraud_xgb.predict_proba(X_test)[:, 1]

In [52]:
# Calculate XGBoost metrics
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_prob)
xgb_pr_auc = average_precision_score(y_test, xgb_prob)

print("Accuracy :", round(xgb_accuracy, 4))
print("Precision:", round(xgb_precision, 4))
print("Recall   :", round(xgb_recall, 4))
print("F1 Score :", round(xgb_f1, 4))
print("ROC-AUC  :", round(xgb_auc, 4))
print("PR-AUC   :", round(xgb_pr_auc, 4))

Accuracy : 0.8204
Precision: 0.038
Recall   : 0.2293
F1 Score : 0.0652
ROC-AUC  : 0.5571
PR-AUC   : 0.0365


In [53]:
# Compare Logistic Regression, Random Forest and XGBoost
fraud_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        lr_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],

    "Precision": [
        lr_precision,
        rf_precision,
        xgb_precision
    ],

    "Recall": [
        lr_recall,
        rf_recall,
        xgb_recall
    ],

    "F1 Score": [
        lr_f1,
        rf_f1,
        xgb_f1
    ],

    "ROC-AUC": [
        lr_auc,
        rf_auc,
        xgb_auc
    ],

    "PR-AUC": [
        lr_pr_auc,
        rf_pr_auc,
        xgb_pr_auc
    ]
})

fraud_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,Logistic Regression,0.620267,0.033039,0.456098,0.061614,0.561648,0.041498
1,Random Forest,0.925467,0.043814,0.082927,0.057336,0.565399,0.036013
2,XGBoost,0.820400,0.038026,0.229268,0.065232,0.557149,0.036515


In [54]:
# Test different fraud probability thresholds
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]

results = []

for threshold in thresholds:

    # Convert fraud probability into final class
    pred = (xgb_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, zero_division=0
        ),
        "F1": f1_score(
            y_test, pred, zero_division=0
        )
    })

threshold_results = pd.DataFrame(results)

threshold_results

,Threshold,Precision,Recall,F1
0,0.1,0.027808,0.995122,0.054104
1,0.2,0.028352,0.917073,0.055003
2,0.3,0.029977,0.741463,0.057625
3,0.4,0.030907,0.453659,0.057872
4,0.5,0.038026,0.229268,0.065232


In [55]:
# Create confusion matrix for fraud predictions
fraud_cm = confusion_matrix(
    y_test,
    xgb_pred
)

print(fraud_cm)

[[12212  2378]
 [  316    94]]


In [56]:
# Get final feature names after preprocessing
feature_names = (
    fraud_xgb
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get XGBoost feature importance values
importance = (
    fraud_xgb
    .named_steps["model"]
    .feature_importances_
)

# Create feature importance table
fraud_feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

# Sort most important features first
fraud_feature_importance = fraud_feature_importance.sort_values(
    "Importance",
    ascending=False
)

fraud_feature_importance.head(15)

,Feature,Importance
0,num__CLAIM_AMOUNT,0.021648
56,cat__POLICY_TYPE_Motor,0.020297
6,num__SUM_INSURED,0.018803
37,cat__OCCUPATION_Salaried,0.018242
55,cat__POLICY_TYPE_Home,0.017951
15,cat__CLAIM_TYPE_Travel Disruption,0.017755
46,cat__STATE_Tamil Nadu,0.017657
66,cat__RISK_BAND_Low,0.017571
26,cat__GENDER_Female,0.017139
42,cat__STATE_Kerala,0.016787


In [57]:
# Select threshold with highest F1 score
best_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

best_fraud_threshold = best_row["Threshold"]

print("Best Threshold:", best_fraud_threshold)
print(best_row)

Best Threshold: 0.5
Threshold    0.500000
Precision    0.038026
Recall       0.229268
F1           0.065232
Name: 4, dtype: float64


In [58]:
# Create final predictions using selected threshold
final_fraud_pred = (
    xgb_prob >= best_fraud_threshold
).astype(int)

# Final evaluation
print(
    classification_report(
        y_test,
        final_fraud_pred,
        target_names=["Normal", "Fraud"]
    )
)

              precision    recall  f1-score   support

      Normal       0.97      0.84      0.90     14590
       Fraud       0.04      0.23      0.07       410

    accuracy                           0.82     15000
   macro avg       0.51      0.53      0.48     15000
weighted avg       0.95      0.82      0.88     15000



In [59]:
# Import model serialization library
import joblib

# Create model directory
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save complete preprocessing + XGBoost model
joblib.dump(
    fraud_xgb,
    MODEL_DIR / "fraud_xgboost_model.joblib"
)

# Save selected fraud threshold
joblib.dump(
    best_fraud_threshold,
    MODEL_DIR / "fraud_threshold.joblib"
)

print("Fraud model saved successfully")

Fraud model saved successfully
